# Quantum Sustainability Challenge 2026 — EDA & Classical Baselines

## What This Notebook Does
1. **Explores all 4 datasets** — understand what we're working with
2. **Builds classical ML baselines** — so we can compare quantum models against them later
3. **Prepares data** for quantum model input

## Key Concepts (Simple Terms)
- **EDA (Exploratory Data Analysis)**: Looking at the data before building models — like reading the map before driving
- **Classification**: Predicting a YES/NO answer (will a fire start today?)
- **Time Series**: Data that changes over time (insurance premiums year by year)
- **Baseline**: A simple model we build first, so we know if our fancy quantum model is actually better

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)

DATA_DIR = "../data/"
print("Libraries loaded successfully!")

---
## Part 1: Wildfire Dataset 1 — Daily Weather & Fire Data (1984–2025)

**What is this?** A daily record of California's weather and whether a wildfire started that day.

**Why it matters:** This is our **primary dataset for Task 1A** — we need to predict `FIRE_START_DAY` (True/False).

**Key columns:**
- `PRECIPITATION`, `MAX_TEMP`, `MIN_TEMP`, `AVG_WIND_SPEED` — weather features
- `FIRE_START_DAY` — **our target** (did a fire start? True/False)
- `TEMP_RANGE`, `WIND_TEMP_RATIO` — pre-engineered features
- `SEASON`, `MONTH`, `DAY_OF_YEAR` — time features

In [ ]:
df1 = pd.read_csv(DATA_DIR + "wildfire_weather_daily.csv")
print(f"Shape: {df1.shape} — {df1.shape[0]} days, {df1.shape[1]} features")
print(f"Date range: {df1['DATE'].min()} to {df1['DATE'].max()}")
print(f"\nColumn types:\n{df1.dtypes}")
print(f"\nMissing values:\n{df1.isnull().sum()[df1.isnull().sum() > 0]}")
df1.head()

### How imbalanced is the target?
**Class imbalance** = when one outcome is much rarer than the other. If fires only happen 5% of the time, a model that always says "no fire" would be 95% accurate — but useless! We need to know this upfront.

In [ ]:
fire_counts = df1['FIRE_START_DAY'].value_counts()
print("Fire Start Day distribution:")
print(fire_counts)
print(f"\nFire days: {fire_counts[True]} ({fire_counts[True]/len(df1)*100:.1f}%)")
print(f"No-fire days: {fire_counts[False]} ({fire_counts[False]/len(df1)*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fire_counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title("Fire vs No-Fire Days (Full Dataset)")
axes[0].set_xticklabels(['No Fire', 'Fire'], rotation=0)
axes[0].set_ylabel("Count")

# Focus on 2018-2021 (competition training period)
df1_train = df1[(df1['YEAR'] >= 2018) & (df1['YEAR'] <= 2021)]
fc_train = df1_train['FIRE_START_DAY'].value_counts()
fc_train.plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'])
axes[1].set_title("Fire vs No-Fire Days (2018-2021 only)")
axes[1].set_xticklabels(['No Fire', 'Fire'], rotation=0)
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.savefig("../results/fire_class_balance.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"\n2018-2021: {fc_train[True]} fire days out of {len(df1_train)} ({fc_train[True]/len(df1_train)*100:.1f}%)")

### When do fires happen? Seasonal patterns
Fires are strongly seasonal — hot, dry summers with wind cause fires. Let's see if the data confirms this.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Monthly fire frequency
monthly = df1.groupby('MONTH')['FIRE_START_DAY'].mean() * 100
monthly.plot(kind='bar', ax=axes[0], color='coral')
axes[0].set_title("% of Days with Fire by Month")
axes[0].set_ylabel("% Fire Days")
axes[0].set_xlabel("Month")

# Seasonal fire frequency
season_order = ['Winter', 'Spring', 'Summer', 'Fall']
seasonal = df1.groupby('SEASON')['FIRE_START_DAY'].mean().reindex(season_order) * 100
seasonal.plot(kind='bar', ax=axes[1], color='orangered')
axes[1].set_title("% of Days with Fire by Season")
axes[1].set_ylabel("% Fire Days")
axes[1].set_xticklabels(season_order, rotation=0)
plt.tight_layout()
plt.savefig("../results/fire_seasonality.png", dpi=150, bbox_inches='tight')
plt.show()

### Feature Correlations
**Correlation** = how strongly two features move together (-1 to +1). High correlation with FIRE_START_DAY tells us which features matter most.

In [ ]:
df1_numeric = df1.select_dtypes(include=[np.number]).copy()
df1_numeric['FIRE_START_DAY'] = df1['FIRE_START_DAY'].astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation with target
corr_target = df1_numeric.corr()['FIRE_START_DAY'].drop('FIRE_START_DAY').sort_values()
corr_target.plot(kind='barh', ax=axes[0], color=['steelblue' if v < 0 else 'coral' for v in corr_target])
axes[0].set_title("Correlation of Each Feature with FIRE_START_DAY")
axes[0].axvline(x=0, color='black', linewidth=0.5)

# Full correlation heatmap
sns.heatmap(df1_numeric.corr(), annot=True, fmt=".2f", cmap='RdBu_r', center=0,
            ax=axes[1], annot_kws={"size": 7})
axes[1].set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("../results/fire_correlations.png", dpi=150, bbox_inches='tight')
plt.show()

---
## Part 2: Wildfire Dataset 2 — County-Level Fire & Weather (2008–2020)

**What is this?** Monthly records per California county — including actual fire names, causes, and acres burned.

**Why it matters:** Gives us **geographic granularity** (county-level) and **fire severity** (acres burned) that Dataset 1 lacks. We'll need this to make ZIP-code-level predictions.

In [ ]:
df2 = pd.read_csv(DATA_DIR + "wildfire_county_monthly.csv")
print(f"Shape: {df2.shape}")
print(f"Date range: {df2['date'].min()} to {df2['date'].max()}")
print(f"Unique counties: {df2['county'].nunique()}")
print(f"\nFire vs No-Fire rows:")
print(f"  Actual fires: {(df2['FIRE_NAME'] != 'no_fire').sum()}")
print(f"  No fire:      {(df2['FIRE_NAME'] == 'no_fire').sum()}")

# Top 10 counties by fire count
fires_only = df2[df2['FIRE_NAME'] != 'no_fire']
top_counties = fires_only['county'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top_counties.plot(kind='barh', ax=axes[0], color='coral')
axes[0].set_title("Top 10 Counties by Number of Fire Events")
axes[0].set_xlabel("Number of Fire Events")

# Largest fires
top_fires = fires_only.nlargest(10, 'GIS_ACRES')[['FIRE_NAME', 'county', 'date', 'GIS_ACRES']]
axes[1].barh(top_fires['FIRE_NAME'] + ' (' + top_fires['county'].str.replace(' County','') + ')',
             top_fires['GIS_ACRES'], color='darkred')
axes[1].set_title("Top 10 Largest Fires by Acres Burned")
axes[1].set_xlabel("Acres")
plt.tight_layout()
plt.savefig("../results/county_fire_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

---
## Part 3: Insurance Datasets — ZIP-Code Level Premiums (2018–2021)

**What is this?** For each California ZIP code, how much insurance premium was collected, how many claims were filed, and the fire risk score — broken down by policy type.

**Why it matters:** This is our **Task 2 dataset** — we need to predict future `Earned Premium` per ZIP code.

**Key terms:**
- **Earned Premium** = total insurance money collected for a ZIP code (our prediction target)
- **Cov A** = Coverage A = the building/dwelling itself
- **Cov C** = Coverage C = personal property inside
- **CAT** = Catastrophe losses (big wildfire events)
- **Non-CAT** = Regular fire losses (smaller incidents)
- **Fire Risk Score** = insurer-assigned risk level (higher = more dangerous)

In [ ]:
def load_insurance_sheet(filepath, sheet_name, year):
    """Load a single ZIP-code sheet from the insurance XLS file."""
    df = pd.read_excel(filepath, sheet_name=sheet_name, header=1, engine='openpyxl')
    first_col = df.columns[0]
    df = df.rename(columns={first_col: 'ZIP_Code'})
    df = df.dropna(subset=['ZIP_Code'])
    df['ZIP_Code'] = pd.to_numeric(df['ZIP_Code'], errors='coerce')
    df = df.dropna(subset=['ZIP_Code'])
    df['ZIP_Code'] = df['ZIP_Code'].astype(int)
    df['Year'] = year
    # Clean column names
    df.columns = [c.replace('\n', ' ').strip() for c in df.columns]
    return df

# Load Homeowners (HO) data for all 4 years — this is the main policy type
ho_2018 = load_insurance_sheet(DATA_DIR + "insurance_2018_2019.XLS", "2018HO", 2018)
ho_2019 = load_insurance_sheet(DATA_DIR + "insurance_2018_2019.XLS", "2019HO", 2019)
ho_2020 = load_insurance_sheet(DATA_DIR + "insurance_2020_2021.XLS", "2020HO", 2020)
ho_2021 = load_insurance_sheet(DATA_DIR + "insurance_2020_2021.XLS", "2021HO", 2021)

insurance_ho = pd.concat([ho_2018, ho_2019, ho_2020, ho_2021], ignore_index=True)
print(f"Combined Homeowners data: {insurance_ho.shape}")
print(f"Unique ZIP codes: {insurance_ho['ZIP_Code'].nunique()}")
print(f"Years: {sorted(insurance_ho['Year'].unique())}")
print(f"\nColumns:\n{list(insurance_ho.columns)}")
insurance_ho.head(3)

In [ ]:
# Premium trends over time
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Total premium by year
yearly_premium = insurance_ho.groupby('Year')['Earned Premium'].sum() / 1e9
yearly_premium.plot(kind='bar', ax=axes[0], color='seagreen')
axes[0].set_title("Total Earned Premium by Year (Homeowners)")
axes[0].set_ylabel("Premium ($ Billions)")
axes[0].set_xticklabels(yearly_premium.index, rotation=0)

# Distribution of Avg Fire Risk Score
risk_col = [c for c in insurance_ho.columns if 'Fire Risk Score' in c and 'Number' not in c][0]
insurance_ho[risk_col] = pd.to_numeric(insurance_ho[risk_col], errors='coerce')
insurance_ho[risk_col].dropna().hist(bins=50, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title("Distribution of Avg Fire Risk Score (All ZIPs)")
axes[1].set_xlabel("Fire Risk Score")

# Premium vs Fire Risk Score
sample = insurance_ho.dropna(subset=[risk_col, 'Earned Premium'])
sample['Earned Premium'] = pd.to_numeric(sample['Earned Premium'], errors='coerce')
sample = sample.dropna(subset=['Earned Premium'])
axes[2].scatter(sample[risk_col], sample['Earned Premium'], alpha=0.2, s=5, c='teal')
axes[2].set_title("Earned Premium vs Fire Risk Score")
axes[2].set_xlabel("Avg Fire Risk Score")
axes[2].set_ylabel("Earned Premium ($)")

plt.tight_layout()
plt.savefig("../results/insurance_overview.png", dpi=150, bbox_inches='tight')
plt.show()

---
## Part 4: Classical ML Baselines for Wildfire Prediction (Task 1A)

### Why build classical baselines first?
The competition (Task 1B) asks us to **compare quantum vs classical**. We need strong classical models to compare against. Also, if quantum is hard, we still have working results.

### Concepts:
- **Random Forest**: Imagine 100 decision trees each voting "fire" or "no fire" — the majority wins. Good for tabular data.
- **XGBoost**: Builds trees one at a time, where each new tree focuses on fixing mistakes the previous trees made. Often the best classical model.
- **Train/Test Split**: We train on 2018-2020 data and test on 2021 data to simulate real prediction.
- **F1 Score**: Balances precision and recall — better than accuracy when classes are imbalanced.
- **AUC-ROC**: How well the model separates fire days from no-fire days. 1.0 = perfect, 0.5 = random guessing.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from xgboost import XGBClassifier

# --- Prepare data: use 2018-2021 as the competition requires ---
df_ml = df1[(df1['YEAR'] >= 2018) & (df1['YEAR'] <= 2021)].copy()
df_ml['FIRE_START_DAY'] = df_ml['FIRE_START_DAY'].astype(int)

features = ['PRECIPITATION', 'MAX_TEMP', 'MIN_TEMP', 'AVG_WIND_SPEED',
            'TEMP_RANGE', 'WIND_TEMP_RATIO', 'MONTH', 'DAY_OF_YEAR',
            'LAGGED_PRECIPITATION', 'LAGGED_AVG_WIND_SPEED']
target = 'FIRE_START_DAY'

df_ml = df_ml.dropna(subset=features)

# Time-based split: train on 2018-2020, test on 2021
train = df_ml[df_ml['YEAR'] <= 2020]
test = df_ml[df_ml['YEAR'] == 2021]

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

# Scale features (important for SVM and quantum models later)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples ({y_train.sum()} fire days)")
print(f"Test set:     {X_test.shape[0]} samples ({y_test.sum()} fire days)")
print(f"Features:     {len(features)}")

In [ ]:
import time

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, scale_pos_weight=y_train.value_counts()[0]/y_train.value_counts()[1],
                              eval_metric='logloss', random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42),
}

results = {}
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train_scaled, y_train)
    train_time = time.time() - t0

    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else y_pred

    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    results[name] = {'F1': f1, 'AUC-ROC': auc, 'Train Time (s)': round(train_time, 3),
                      'Predictions': y_pred, 'Probabilities': y_prob}
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"  F1 Score: {f1:.4f} | AUC-ROC: {auc:.4f} | Train: {train_time:.3f}s")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=['No Fire', 'Fire']))

In [ ]:
# Summary comparison plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

names = list(results.keys())
f1s = [results[n]['F1'] for n in names]
aucs = [results[n]['AUC-ROC'] for n in names]
times = [results[n]['Train Time (s)'] for n in names]

axes[0].barh(names, f1s, color='steelblue')
axes[0].set_title("F1 Score (higher = better)")
axes[0].set_xlim(0, 1)
for i, v in enumerate(f1s):
    axes[0].text(v + 0.01, i, f'{v:.3f}')

axes[1].barh(names, aucs, color='coral')
axes[1].set_title("AUC-ROC (higher = better)")
axes[1].set_xlim(0, 1)
for i, v in enumerate(aucs):
    axes[1].text(v + 0.01, i, f'{v:.3f}')

axes[2].barh(names, times, color='seagreen')
axes[2].set_title("Training Time (seconds)")
for i, v in enumerate(times):
    axes[2].text(v + 0.001, i, f'{v:.3f}s')

plt.suptitle("Classical Baseline Comparison — Wildfire Prediction", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("../results/classical_baseline_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# Save results for later quantum comparison
import json
baseline_summary = {n: {'F1': results[n]['F1'], 'AUC-ROC': results[n]['AUC-ROC'],
                         'Train Time (s)': results[n]['Train Time (s)']} for n in names}
with open('../results/classical_baselines.json', 'w') as f:
    json.dump(baseline_summary, f, indent=2)
print("\nClassical baselines saved to results/classical_baselines.json")

### Feature Importance from Random Forest
Which weather features matter most for predicting wildfires? This tells us which features to feed into our quantum model (we can only use ~6-10 features due to qubit limits).

In [ ]:
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values()

plt.figure(figsize=(10, 5))
importances.plot(kind='barh', color='teal')
plt.title("Random Forest Feature Importance — Which Weather Features Predict Fires?")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("../results/feature_importance.png", dpi=150, bbox_inches='tight')
plt.show()

# Top features for quantum model (we'll use the top 6-8)
top_features = importances.tail(8).index.tolist()
print(f"\nTop 8 features for quantum model: {top_features}")

---
## Part 5: Quantum Machine Learning — Variational Quantum Classifier (Task 1A)

### What is Quantum Machine Learning? (Simple Explanation)

**Classical ML** uses math on regular bits (0s and 1s) to find patterns.

**Quantum ML** uses **qubits** — which can be 0, 1, or *both at once* (superposition). This lets us explore many possibilities simultaneously.

### How a Variational Quantum Classifier (VQC) works:

```
Step 1: ENCODE your data into a quantum circuit
        (turn numbers into qubit rotations — like turning the dial on a radio)

Step 2: PROCESS through a parameterized circuit (ansatz)
        (quantum gates with adjustable knobs — like neural network weights)

Step 3: MEASURE the qubits
        (collapse quantum state → get a 0 or 1 → "no fire" or "fire")

Step 4: OPTIMIZE
        (a classical computer adjusts the knobs to reduce prediction error)

Repeat Steps 1-4 until the model learns the pattern.
```

### Key Quantum Concepts:

- **Qubit**: A quantum bit. Unlike classical bits (0 or 1), qubits can be in superposition (both at once).
- **Feature Map**: How we encode our weather data into quantum states. We use **angle encoding**: each feature becomes a rotation angle on a qubit. 6 features → 6 qubits.
- **Ansatz**: The trainable quantum circuit. Think of it as the "brain" of the model. We use **EfficientSU2** — a circuit with rotation gates and entangling connections.
- **Entanglement**: Qubits linked so that the state of one affects the other — this is what gives quantum computing its power.
- **Shots**: How many times we run the circuit to get statistics (quantum is probabilistic).
- **Simulator**: We run on a classical computer that *simulates* a quantum computer (using Qiskit Aer).

### Why only 6-8 features?
Each feature needs ~1 qubit. Simulating N qubits takes 2^N memory. At 20 qubits, that's ~1 million states. At 30 qubits, ~1 billion. So we keep it small and use the best features from our Random Forest analysis.

In [ ]:
# --- Step 1: Prepare data for quantum model ---
# Use top features identified by Random Forest
# Quantum circuits work best with fewer features, so we pick 6
N_QUBITS = 6
quantum_features = importances.tail(N_QUBITS).index.tolist()
print(f"Using {N_QUBITS} features (= {N_QUBITS} qubits): {quantum_features}")

X_train_q = scaler.fit_transform(train[quantum_features])
X_test_q = scaler.transform(test[quantum_features])

# Scale to [0, pi] range for angle encoding
from sklearn.preprocessing import MinMaxScaler
pi_scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train_q = pi_scaler.fit_transform(X_train_q)
X_test_q = pi_scaler.transform(X_test_q)

# Use a smaller training subset for quantum (simulators are slow)
# We'll use 200 training samples and 50 test samples
np.random.seed(42)
train_idx = np.random.choice(len(X_train_q), size=200, replace=False)
test_idx = np.random.choice(len(X_test_q), size=50, replace=False)

X_train_qml = X_train_q[train_idx]
y_train_qml = y_train.values[train_idx]
X_test_qml = X_test_q[test_idx]
y_test_qml = y_test.values[test_idx]

print(f"\nQuantum training set: {X_train_qml.shape}")
print(f"Quantum test set: {X_test_qml.shape}")
print(f"Fire days in quantum train: {y_train_qml.sum()}/{len(y_train_qml)}")
print(f"Fire days in quantum test: {y_test_qml.sum()}/{len(y_test_qml)}")

### Step 2: Build the Quantum Circuit

We build two pieces:
1. **Feature Map** (ZZFeatureMap) — encodes our 6 weather features into 6 qubits using rotation gates + entanglement
2. **Ansatz** (EfficientSU2) — the trainable part with adjustable parameters

Think of it like a neural network:
- Feature Map = input layer (how data enters)
- Ansatz = hidden layers (where learning happens)
- Measurement = output layer (the prediction)

In [ ]:
from qiskit.circuit.library import ZZFeatureMap, EfficientSU2
from qiskit_machine_learning.algorithms import VQC
from qiskit_aer import AerSimulator
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.optimizers import COBYLA

# Feature Map: encodes data into quantum states
# ZZFeatureMap applies rotations based on feature values AND entangles pairs of qubits
# This captures interactions between features (e.g., temperature AND wind together)
feature_map = ZZFeatureMap(feature_dimension=N_QUBITS, reps=1)

# Ansatz: the trainable circuit
# EfficientSU2 uses rotation gates (Ry, Rz) and CNOT entangling gates
# reps=2 means 2 layers of rotations+entanglement (more layers = more expressive but slower)
ansatz = EfficientSU2(num_qubits=N_QUBITS, reps=2, entanglement='linear')

print(f"Feature Map: {feature_map.num_qubits} qubits, depth {feature_map.depth()}")
print(f"Ansatz: {ansatz.num_qubits} qubits, depth {ansatz.depth()}, {ansatz.num_parameters} trainable parameters")
print(f"\nTotal circuit depth: {feature_map.depth() + ansatz.depth()}")

# Visualize the circuits
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
feature_map.decompose().draw('mpl', ax=axes[0])
axes[0].set_title("Feature Map (ZZFeatureMap) — How data enters the quantum circuit")
ansatz.decompose().draw('mpl', ax=axes[1])
axes[1].set_title("Ansatz (EfficientSU2) — Trainable quantum circuit")
plt.tight_layout()
plt.savefig("../results/quantum_circuits.png", dpi=150, bbox_inches='tight')
plt.show()

### Step 3: Train the VQC

Now we train the quantum model. The **COBYLA optimizer** adjusts the circuit parameters to minimize classification error — similar to how gradient descent trains a neural network.

**Note:** This runs on a simulator (Qiskit Aer) on your laptop. Each training step evaluates the quantum circuit many times ("shots"). This is slower than classical models — that's expected and important to document for your submission.

In [ ]:
optimizer = COBYLA(maxiter=80)

vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer,
    sampler=StatevectorSampler(),
)

print("Training VQC... (this may take a few minutes on the simulator)")
print(f"  Qubits: {N_QUBITS}")
print(f"  Trainable parameters: {ansatz.num_parameters}")
print(f"  Optimizer: COBYLA, max iterations: 80")
print(f"  Training samples: {len(X_train_qml)}")

t0 = time.time()
vqc.fit(X_train_qml, y_train_qml)
quantum_train_time = time.time() - t0
print(f"\nTraining complete in {quantum_train_time:.1f} seconds")

### Step 4: Evaluate Quantum vs Classical (Task 1B Preview)

In [ ]:
# Quantum predictions
y_pred_q = vqc.predict(X_test_qml)
f1_q = f1_score(y_test_qml, y_pred_q)

print("="*60)
print("  QUANTUM VQC RESULTS")
print(f"  F1 Score: {f1_q:.4f}")
print(f"  Training Time: {quantum_train_time:.1f}s")
print(f"  Qubits: {N_QUBITS} | Ansatz layers: 2 | Parameters: {ansatz.num_parameters}")
print("="*60)
print(classification_report(y_test_qml, y_pred_q, target_names=['No Fire', 'Fire']))

# Compare with classical models on the SAME subset
print("\n--- Comparison on same data subset ---")
comparison = {'VQC (Quantum)': {'F1': f1_q, 'Train Time (s)': round(quantum_train_time, 1)}}

for name, model in models.items():
    model.fit(X_train_q[train_idx], y_train_qml)
    y_pred_c = model.predict(X_test_q[test_idx])
    f1_c = f1_score(y_test_qml, y_pred_c)
    comparison[name] = {'F1': f1_c, 'Train Time (s)': round(time.time() - time.time(), 3)}
    print(f"  {name}: F1 = {f1_c:.4f}")

print(f"\n  VQC (Quantum): F1 = {f1_q:.4f}")

---
## Part 6: Quantum Kernel Method (Alternative Approach for Task 1A)

### What is a Quantum Kernel?

A **kernel** is a function that measures how *similar* two data points are. In classical SVMs, we use the RBF kernel.

A **quantum kernel** computes this similarity using quantum circuits:
1. Encode data point A into qubits
2. Apply the *inverse* encoding of data point B
3. Measure — if the result is close to the initial state, A and B are similar

**Why try this?** Quantum kernels don't have the "barren plateau" problem that VQCs have (where gradients vanish and training stalls). They can capture complex feature interactions that classical kernels miss.

In [ ]:
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit.primitives import StatevectorSampler as Sampler

# Use the same feature map as before
kernel_feature_map = ZZFeatureMap(feature_dimension=N_QUBITS, reps=1)
quantum_kernel = FidelityQuantumKernel(feature_map=kernel_feature_map)

# Compute quantum kernel matrix (this computes pairwise similarity between all training samples)
print("Computing quantum kernel matrix... (this may take a minute)")
t0 = time.time()
train_kernel_matrix = quantum_kernel.evaluate(X_train_qml)
test_kernel_matrix = quantum_kernel.evaluate(X_test_qml, X_train_qml)
kernel_time = time.time() - t0
print(f"Kernel computation took {kernel_time:.1f}s")

# Use this quantum kernel with a classical SVM
from sklearn.svm import SVC
qsvm = SVC(kernel='precomputed', class_weight='balanced')
qsvm.fit(train_kernel_matrix, y_train_qml)
y_pred_qk = qsvm.predict(test_kernel_matrix)

f1_qk = f1_score(y_test_qml, y_pred_qk)
print(f"\nQuantum Kernel SVM F1 Score: {f1_qk:.4f}")
print(classification_report(y_test_qml, y_pred_qk, target_names=['No Fire', 'Fire']))

---
## Part 7: Final Summary — All Models Compared

This is the heart of **Task 1B**: a fair comparison of quantum vs classical approaches.

In [ ]:
# Final comparison table
summary_data = []
for name, model in models.items():
    model.fit(X_train_q[train_idx], y_train_qml)
    t0 = time.time()
    y_p = model.predict(X_test_q[test_idx])
    pred_time = time.time() - t0
    summary_data.append({
        'Model': name, 'Type': 'Classical',
        'F1': round(f1_score(y_test_qml, y_p), 4),
        'Qubits': '-', 'Parameters': '-',
    })

summary_data.append({
    'Model': 'VQC (Quantum)', 'Type': 'Quantum',
    'F1': round(f1_q, 4),
    'Qubits': N_QUBITS, 'Parameters': ansatz.num_parameters,
})
summary_data.append({
    'Model': 'Quantum Kernel SVM', 'Type': 'Quantum',
    'F1': round(f1_qk, 4),
    'Qubits': N_QUBITS, 'Parameters': '-',
})

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*70)
print("  COMPLETE MODEL COMPARISON (Task 1B)")
print("="*70)
print(summary_df.to_string(index=False))

# Save
summary_df.to_csv('../results/model_comparison.csv', index=False)
print("\nSaved to results/model_comparison.csv")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue' if t == 'Classical' else 'coral' for t in summary_df['Type']]
ax.barh(summary_df['Model'], summary_df['F1'], color=colors)
ax.set_xlabel('F1 Score')
ax.set_title('Quantum vs Classical — Wildfire Prediction F1 Scores')
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
for i, v in enumerate(summary_df['F1']):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center')
ax.legend(['Classical', 'Quantum'], loc='lower right')
plt.tight_layout()
plt.savefig("../results/quantum_vs_classical.png", dpi=150, bbox_inches='tight')
plt.show()

---
## What We've Accomplished

1. **EDA** — Explored all 4 datasets, understood fire seasonality, class imbalance, insurance premium trends
2. **Classical Baselines** — Trained Logistic Regression, Random Forest, XGBoost, SVM on wildfire data
3. **Feature Selection** — Identified top 6-8 features for quantum model using Random Forest importance
4. **Quantum VQC** — Built and trained a Variational Quantum Classifier with 6 qubits on Qiskit Aer
5. **Quantum Kernel SVM** — Implemented a second quantum approach using quantum kernel methods
6. **Task 1B Preview** — Compared quantum vs classical on the same data subset

## Next Steps (in separate notebooks)
- **Task 2**: QML time series model for insurance premium prediction
- **Optimization**: More training iterations, hyperparameter tuning, try on AWS Braket
- **Visualization**: Geospatial risk maps for California ZIP codes
- **Submission PDF**: Write up results following the required structure